In [ ]:

# LOQBOX DATA CHALLENGE - SOLUÇÃO COMPLETA
# Autor: Jonatas de Siqueira
# Data: 27 de Maio 2026


# Instalar dependências
!pip install mysql-connector-python pandas matplotlib seaborn numpy scipy -q

# Importar as bibliotecas
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURAÇÃO DA CONEXÃO COM O BANCO

# Configuração do banco de dados conforme email enviado
DB_CONFIG = {
   'host': '*************',  # Insira o IP disponibilizado 
    'user': '************',   # Insira o Usuario 
    'password': '************',  # Insira a Senha
    'database': '************',   # Insira o Local
    'port': ************      # Insira a porta
}
# A fim de manter os dados sensiveis preservados de acordo com a LGPD os dados foram substituidos por *** e 
# para testa-los, deve-ra inserir as credenciais, ou solicita-los. 


def get_connection():
    """Cria e retorna conexão com o banco MySQL."""
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        print("✅ Você esta conectado ao banco MySQL com sucesso, seus dados estao atualizados!")
        return conn
    except mysql.connector.Error as err:
        print(f"❌ Erro de conexão, verifique as credenciais digitadas: {err}")
        raise

def run_query(query, params=None):
    """Executa uma query e retorna um DataFrame."""
    conn = get_connection()
    try:
        if params:
            return pd.read_sql(query, conn, params=params)
        return pd.read_sql(query, conn)
    finally:
        conn.close()


In [ ]:

# Pergunta 1: Quais os 10 produtos mais caros da empresa?

print("\n PERGUNTA 1: Os 10 produtos mais caros?")

query_1 = """
SELECT
    PRODUCT_COD,
    PRODUCT_NAME,
    PRODUCT_VAL,
    DEP_NAME,
    SECTION_NAME
FROM data_product
ORDER BY PRODUCT_VAL DESC
LIMIT 10
"""

df_q1 = run_query(query_1)
print("\n✅ Os 10 produtos mais caros dentro da empresa são:")
print(df_q1.to_string(index=False))

In [ ]:

# Pergunta 2: Quais seções os departamentos 'BEBIDAS' e 'PADARIA' possuem?

print("\n\n PERGUNTA 2: Seções dos departamentos 'BEBIDAS' e 'PADARIA'")

query_2 = """
SELECT
    DEP_NAME,
    SECTION_NAME,
    SECTION_COD
FROM data_product
WHERE DEP_NAME IN ('BEBIDAS', 'PADARIA')
GROUP BY DEP_NAME, SECTION_NAME, SECTION_COD
ORDER BY DEP_NAME, SECTION_NAME
"""

df_q2 = run_query(query_2)
print("\n✅ As seções encontradas foram:")
print(df_q2.to_string(index=False))

In [ ]:

# Pergunta 3: Total de vendas por Business Area no 1º trimestre de 2019

print("\n\n PERGUNTA 3: Total de vendas por Área de Negócio)")

query_3 = """
SELECT
    ds.BUSINESS_NAME,
    SUM(dss.SALES_VALUE) AS TOTAL_SALES
FROM data_store_sales dss
INNER JOIN data_store_cad ds ON dss.STORE_CODE = ds.STORE_CODE
WHERE dss.DATE BETWEEN '2019-01-01' AND '2019-03-31'
GROUP BY ds.BUSINESS_NAME
ORDER BY TOTAL_SALES DESC
"""

df_q3 = run_query(query_3)
print("\n✅ 3.	Total vendido no primeiro trimestre de 2019 por Business Area:")
print(df_q3.to_string(index=False))



In [ ]:
# 3. FUNÇÃO DINÂMICA: retrieve_data()

print("FUNÇÃO DINÂMICA: retrieve_data()")

def retrieve_data(product_code=None, store_code=None, date=None):
    """
    A Função dinâmica não havia utilizado, tinha conhecimento apenas teórico,
    utilizei material didatico (livro e ebook)para auxilio na montagem,

    utilizei IA para ajustar o primeiro codigo, pois continha erro que nao estava
    conseguindo ajustar, o erro era "" duplas eu estava utilizando '' simples
    e %S que era o correto %s
    """
    query = "SELECT * FROM data_product_sales WHERE 1=1"
    params = []

    # Filtro em product_code
    if product_code is not None:
        if isinstance(product_code, list):
            placeholders = ','.join(['%s'] * len(product_code)) #USEI IA neste ponto para ajsutar a linha que nao consegui ajustar
            query += f" AND PRODUCT_CODE IN ({placeholders})"
            params.extend(product_code)
        else:
            query += " AND PRODUCT_CODE = %s"  # USEI IA até esse ponto
            params.append(product_code)

    # Filtro em store_code
    if store_code is not None:
        if isinstance(store_code, list):
            placeholders = ','.join(['%s'] * len(store_code))
            query += f" AND STORE_CODE IN ({placeholders})"
            params.extend(store_code)
        else:
            query += " AND STORE_CODE = %s"
            params.append(store_code)

    # Filtro em intervalo de data
    if date is not None and len(date) == 2:
        query += " AND DATE BETWEEN %s AND %s"
        params.extend(date)

    # Executar a  query
    conn = get_connection()
    try:
        df = pd.read_sql(query, conn, params=params)
        return df
    finally:
        conn.close()


# Testando a função dinâmica criada
print("\n Testando a função retrieve_data():")

# Teste 1: Produto único
print("\n1️⃣ retrieve_data(product_code=123):")
df_test1 = retrieve_data(product_code=123)
print(f"   → {df_test1.shape[0]} linhas, {df_test1.shape[1]} colunas")

# Teste 2: Intervalo de datas sugeridos
print("\n2️⃣ retrieve_data(date=['2019-01-01', '2019-01-31']):")
df_test2 = retrieve_data(date=['2019-01-01', '2019-01-31'])
print(f"   → {df_test2.shape[0]} linhas, {df_test2.shape[1]} colunas")

# Teste 3: Múltiplos produtos + intervalo de datas dugeridas
print("\n3️⃣ retrieve_data(product_code=[123,456,789], date=['2019-01-01','2019-12-31']):")
df_test3 = retrieve_data(product_code=[123, 456, 789], date=['2019-01-01', '2019-12-31'])
print(f"   → {df_test3.shape[0]} linhas, {df_test3.shape[1]} colunas")

print("\n✅ Função retrieve_data() funcionando corretamente!")


In [ ]:

# 4. QUERIES PRONTAS + VISUALIZAÇÃO DO CLIENTE

print("QUERIES PRONTAS + VISUALIZAÇÃO")

# QUERIES PRONTAS (como fornecidas, SEM MODIFICAÇÕES)
QUERY_STORE_CAD = """
SELECT
    STORE_CODE,
    STORE_NAME,
    START_DATE,
    END_DATE,
    BUSINESS_NAME,
    BUSINESS_CODE
FROM data_store_cad
"""

QUERY_STORE_SALES = """
SELECT
    STORE_CODE,
    DATE,
    SALES_VALUE,
    SALES_QTY
FROM data_store_sales
WHERE DATE BETWEEN '2019-01-01' AND '2019-12-31'
"""

print("\n Carregando dados das queries prontas...")

# Executar queries prontas (SEM MODIFICAÇÕES)
df_store_cad = run_query(QUERY_STORE_CAD)
df_store_sales = run_query(QUERY_STORE_SALES)

print(f"   data_store_cad: {df_store_cad.shape[0]} linhas")
print(f"   data_store_sales: {df_store_sales.shape[0]} linhas")

# APLICAR FILTRO ADICIONAL (conforme instrução)
date_filter = ['2019-10-01', '2019-12-31']

# Garantir que a coluna DATE seja datetime para comparação correta
df_store_sales['DATE'] = pd.to_datetime(df_store_sales['DATE'])

df_store_sales_filtered = df_store_sales[
    (df_store_sales['DATE'] >= date_filter[0]) &
    (df_store_sales['DATE'] <= date_filter[1])
]

print(f"\n Filtro adicional aplicado: {date_filter[0]} a {date_filter[1]}")
print(f"   Linhas filtradas: {df_store_sales_filtered.shape[0]}")

# Mesclar os dataframes
df_combined = df_store_sales_filtered.merge(df_store_cad, on='STORE_CODE', how='left')
print(f"   Dados mesclados: {df_combined.shape[0]} linhas")

# Agregar por Área de Negócio
df_business = df_combined.groupby('BUSINESS_NAME')['SALES_VALUE'].sum().reset_index()
df_business = df_business.sort_values('SALES_VALUE', ascending=False)

print("\n Vendas por Área de Negócio: 2019 (Out a Dez)")
print(df_business.to_string(index=False))

# CRIAR VISUALIZAÇÃO (conforme solicitado)
print("\n Gerando visualização...")

## Utilizei ia para revisar estar parte ate o print pois nao lembrava por completo alguns comandos.
plt.figure(figsize=(12, 7))
bars = plt.barh(df_business['BUSINESS_NAME'], df_business['SALES_VALUE'],
                color='#1898E0', edgecolor='navy', alpha=0.85)

plt.title('Total de Vendas por Área de Negócio - 2019 (Out a Dez)',
          fontsize=14, fontweight='bold', pad=22)
plt.xlabel('Total de Vendas (R$)', fontsize=12, fontweight='semibold')
plt.ylabel('Área de Negócio', fontsize=12, fontweight='semibold')
plt.grid(axis='x', alpha=0.3, linestyle='--')

# Adicionar rótulos de valores
for bar, value in zip(bars, df_business['SALES_VALUE']):
    plt.text(value, bar.get_y() + bar.get_height()/2,
             f'R$ {value:,.0f}', va='center', ha='right', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('q4_2019_sales_by_business.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualização salva como 'q4_2019_sales_by_business.png'")



In [ ]:
# 5. IMDB_MOVIES - VISUALIZAÇÃO PRÓPRIA

print("IMDB_MOVIES - VISUALIZAÇÃO PERSONALIZADA")

print("\n Carregando dados do IMDB_movies...")

query_imdb = "SELECT * FROM IMDB_movies"
df_movies = run_query(query_imdb)

print(f"   Carregados {df_movies.shape[0]} filmes")
print(f"   Colunas: {df_movies.columns.tolist()}")

# Identificar a coluna de rating dinamicamente
rating_col = None
for col in ['rating', 'Rating', 'imdb_rating', 'score', 'averageRating']:
    if col in df_movies.columns:
        rating_col = col
        break

if rating_col is None and len(df_movies.columns) >= 3:
    rating_col = df_movies.columns[2]  # fallback

print(f"   Usando coluna de avaliação: '{rating_col}'")


# Visualização 1: Distribuição das Avaliações (Histograma + KDE)

print("\n Criando gráfico de distribuição das avaliações...")

plt.figure(figsize=(12, 6))

# Histograma com KDE
sns.histplot(df_movies[rating_col].dropna(), bins=35, kde=True,
             color='#6B256F', edgecolor='black', alpha=0.7)

# Linhas de estatísticas
mean_val = df_movies[rating_col].mean()
median_val = df_movies[rating_col].median()

plt.axvline(mean_val, color='red', linestyle='--', linewidth=2,
            label=f'Média: {mean_val:.2f}')
plt.axvline(median_val, color='green', linestyle=':', linewidth=2,
            label=f'Mediana: {median_val:.2f}')

plt.title('Distribuição das Avaliações dos Filmes IMDB', fontsize=14, fontweight='bold')
plt.xlabel('Avaliação', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.legend(loc='upper left')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('imdb_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"   ✅ Estatísticas - Média: {mean_val:.2f}, Mediana: {median_val:.2f}")


# Visualização 2: Top Gêneros (se a coluna de gênero existir)
if 'genre' in df_movies.columns or 'Genre' in df_movies.columns:
    genre_col = 'genre' if 'genre' in df_movies.columns else 'Genre'
    print(f"\n Criando gráfico de Top Gêneros...")

    # Contar filmes por gênero
    genre_counts = df_movies[genre_col].str.split(',').explode().str.strip().value_counts().head(10)

    plt.figure(figsize=(10, 8))
    plt.barh(genre_counts.index, genre_counts.values, color='#D0E4AE', edgecolor='black')
    plt.title('Top 10 Gêneros de Filmes', fontsize=14, fontweight='bold')
    plt.xlabel('Número de Filmes', fontsize=12)
    plt.ylabel('Gênero', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('imdb_top_genres.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   ✅ Gráfico de Top Gêneros salvo")


# Visualização 3: Avaliações por Ano (se a coluna de ano existir)

if 'year' in df_movies.columns or 'Year' in df_movies.columns:
    year_col = 'year' if 'year' in df_movies.columns else 'Year'
    print(f"\n Criando gráfico de Avaliações por Ano...")

    # Limpar dados de ano
    df_movies_year = df_movies[df_movies[year_col] > 1900].copy()
    df_movies_year = df_movies_year[df_movies_year[year_col] <= 2026]

    plt.figure(figsize=(14, 6))
    plt.scatter(df_movies_year[year_col], df_movies_year[rating_col],
                alpha=0.3, color='#35122E', s=20)

    # Adicionar linha de tendência
    from scipy import stats
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        df_movies_year[year_col], df_movies_year[rating_col]
    )
    x_trend = np.array([df_movies_year[year_col].min(), df_movies_year[year_col].max()])
    y_trend = slope * x_trend + intercept
    plt.plot(x_trend, y_trend, color='red', linewidth=2, label=f'Tendência (r={r_value:.2f})')

    plt.title(f'Avaliações de Filmes por Ano (n={len(df_movies_year)} filmes)', fontsize=14, fontweight='bold')
    plt.xlabel('Ano', fontsize=12)
    plt.ylabel('Avaliação', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('imdb_ratings_by_year.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   ✅ Gráfico de Avaliações por Ano salvo")



In [ ]:

# 6. JUSTIFICATIVA DAS VISUALIZAÇÕES

print("JUSTIFICATIVA DAS VISUALIZAÇÕES DO IMDB")

print("""
 POR QUE ESCOLHI ESTAS VISUALIZAÇÕES?

1️⃣ DISTRIBUIÇÃO DAS AVALIAÇÕES (Histograma + KDE):
   - Mostra o padrão geral das notas dos filmes
   - KDE ajuda a visualizar a curva de distribuição suavizada
   - Linhas de média e mediana fornecem contexto estatístico
   - Ajuda a identificar se as avaliações são assimétricas ou normais

2️⃣ TOP GÊNEROS (Gráfico de Barras):
   - Comparação visual clara da popularidade dos gêneros
   - Barras horizontais facilitam a leitura dos rótulos
   - Ajuda times de conteúdo a entender quais gêneros são mais produzidos

3️⃣ AVALIAÇÕES POR ANO (Gráfico de Dispersão + Linha de Tendência):
   - Mostra a evolução temporal da qualidade dos filmes
   - Pontos de dispersão revelam densidade dos dados
   - Linha de tendência com coeficiente de correlação (r) mostra se as notas estão melhorando ou piorando
   - Útil para análise de tendências da indústria

Estas visualizações demonstram storytelling com dados, compreensão estatística
e a capacidade de extrair insights acionáveis de dados brutos.
""")



In [ ]:
# 7. RESUMO FINAL

print("RESUMO DA SOLUÇÃO")

print("""
✅ PERGUNTAS SQL CONCLUÍDAS:
   - Q1: Top 10 produtos mais caros
   - Q2: Seções dos departamentos BEBIDAS e PADARIA
   - Q3: Total de vendas por Área de Negócio no Q4 2019

✅ FUNÇÃO DINÂMICA CONCLUÍDA:
   - retrieve_data(product_code, store_code, date)
   - Suporta valores únicos, listas e parâmetros opcionais
   - Retorna todas as colunas da tabela data_product_sales

✅ QUERIES PRONTAS CONCLUÍDAS:
   - Queries usadas como fornecidas (sem modificações)
   - Filtro adicional aplicado: ['2019-10-01', '2019-12-31']
   - Gráfico de barras gerado para vendas por Área de Negócio

✅ VISUALIZAÇÕES IMDB CONCLUÍDAS:
   - Distribuição das avaliações (histograma + KDE)
   - Top gêneros (gráfico de barras)
   - Avaliações por ano (dispersão + linha de tendência)

✅ ARQUIVOS GERADOS:
   - q4_2019_sales_by_business.png
   - imdb_rating_distribution.png
   - imdb_top_genres.png (se coluna de gênero existir)
   - imdb_ratings_by_year.png (se coluna de ano existir)
""")

print("\n LOQBOX DATA CHALLENGE - SOLUÇÃO COMPLETADA COM SUCESSO!")